In [5]:
#hide
from fastbook import *
from IPython.display import display,HTML

In [6]:
from fastai.text.all import *
path = untar_data(URLs.IMDB)

In [7]:
import torch
from fastai.vision.all import *

# Set default device to MPS
defaults.device = torch.device("mps")

In [8]:
print(torch.backends.mps.is_available())

True


In [9]:
print(torch.backends.mps.is_built())

True


In [10]:
path

Path('/Users/aidardarmesh/.fastai/data/imdb')

In [11]:
files = get_text_files(path, folders = ['train', 'test', 'unsup'])

In [12]:
files

(#100000) [Path('/Users/aidardarmesh/.fastai/data/imdb/test/neg/1821_4.txt'),Path('/Users/aidardarmesh/.fastai/data/imdb/test/neg/9487_1.txt'),Path('/Users/aidardarmesh/.fastai/data/imdb/test/neg/4604_4.txt'),Path('/Users/aidardarmesh/.fastai/data/imdb/test/neg/2828_2.txt'),Path('/Users/aidardarmesh/.fastai/data/imdb/test/neg/10890_1.txt'),Path('/Users/aidardarmesh/.fastai/data/imdb/test/neg/3351_4.txt'),Path('/Users/aidardarmesh/.fastai/data/imdb/test/neg/8070_2.txt'),Path('/Users/aidardarmesh/.fastai/data/imdb/test/neg/1027_4.txt'),Path('/Users/aidardarmesh/.fastai/data/imdb/test/neg/8248_3.txt'),Path('/Users/aidardarmesh/.fastai/data/imdb/test/neg/4290_4.txt'),Path('/Users/aidardarmesh/.fastai/data/imdb/test/neg/10096_1.txt'),Path('/Users/aidardarmesh/.fastai/data/imdb/test/neg/11890_1.txt'),Path('/Users/aidardarmesh/.fastai/data/imdb/test/neg/2008_1.txt'),Path('/Users/aidardarmesh/.fastai/data/imdb/test/neg/472_4.txt'),Path('/Users/aidardarmesh/.fastai/data/imdb/test/neg/9876_2.txt

In [13]:
txt = files[0].open().read(); txt[:75]

'Alan Rickman & Emma Thompson give good performances with southern/New Orlea'

In [14]:
spacy = WordTokenizer()
toks = first(spacy([txt]))
coll_repr(toks, 30)

'(#121) [\'Alan\',\'Rickman\',\'&\',\'Emma\',\'Thompson\',\'give\',\'good\',\'performances\',\'with\',\'southern\',\'/\',\'New\',\'Orleans\',\'accents\',\'in\',\'this\',\'detective\',\'flick\',\'.\',\'It\',"\'s",\'worth\',\'seeing\',\'for\',\'their\',\'scenes-\',\'and\',\'Rickman\',"\'s",\'scene\'...]'

In [15]:
first(spacy(['The U.S. dollar 1.00.']))

(#5) ['The','U.S.','dollar','1.00','.']

In [16]:
tkn = Tokenizer(spacy)
print(coll_repr(tkn(txt), 31))

(#139) ['xxbos','xxmaj','alan','xxmaj','rickman','&','xxmaj','emma','xxmaj','thompson','give','good','performances','with','southern','/','xxmaj','new','xxmaj','orleans','accents','in','this','detective','flick','.','xxmaj','it',"'s",'worth','seeing'...]


In [17]:
defaults.text_proc_rules

[<function fastai.text.core.fix_html(x)>,
 <function fastai.text.core.replace_rep(t)>,
 <function fastai.text.core.replace_wrep(t)>,
 <function fastai.text.core.spec_add_spaces(t)>,
 <function fastai.text.core.rm_useless_spaces(t)>,
 <function fastai.text.core.replace_all_caps(t)>,
 <function fastai.text.core.replace_maj(t)>,
 <function fastai.text.core.lowercase(t, add_bos=True, add_eos=False)>]

In [18]:
coll_repr(tkn('©   Fast.ai www.fast.ai/INDEX'), 31)

"(#11) ['xxbos','©','xxmaj','fast.ai','xxrep','3','w','.fast.ai','/','xxup','index']"

In [19]:
txts = L(o.open().read() for o in files[:2000])

In [20]:
txts[0]

"Alan Rickman & Emma Thompson give good performances with southern/New Orleans accents in this detective flick. It's worth seeing for their scenes- and Rickman's scene with Hal Holbrook. These three actors mannage to entertain us no matter what the movie, it seems. The plot for the movie shows potential, but one gets the impression in watching the film that it was not pulled off as well as it could have been. The fact that it is cluttered by a rather uninteresting subplot and mostly uninteresting kidnappers really muddles things. The movie is worth a view- if for nothing more than entertaining performances by Rickman, Thompson, and Holbrook."

In [21]:
def subword(sz):
    sp = SubwordTokenizer(vocab_sz=sz)
    sp.setup(txts)
    return ' '.join(first(sp([txt]))[:40])

In [22]:
subword(1000)

sentencepiece_trainer.cc(178) LOG(INFO) Running command: --input=tmp/texts.out --vocab_size=1000 --model_prefix=tmp/spm --character_coverage=0.99999 --model_type=unigram --unk_id=9 --pad_id=-1 --bos_id=-1 --eos_id=-1 --minloglevel=2 --user_defined_symbols=▁xxunk,▁xxpad,▁xxbos,▁xxeos,▁xxfld,▁xxrep,▁xxwrep,▁xxup,▁xxmaj --hard_vocab_limit=false


'▁A l an ▁R ick man ▁ & ▁E mm a ▁Th o mp s on ▁give ▁good ▁performance s ▁with ▁so u ther n / N e w ▁O r le an s ▁a c c ent s ▁in'

In [23]:
subword(200)

'▁A l an ▁ R ic k m an ▁ & ▁ E m m a ▁ T h o m p s on ▁ g i ve ▁ g o o d ▁p er f or m an ce'

In [24]:
subword(10_000)

"▁Alan ▁Rick man ▁ & ▁Emma ▁Thompson ▁give ▁good ▁performances ▁with ▁southern / N ew ▁O rleans ▁accents ▁in ▁this ▁detective ▁flick . ▁It ' s ▁worth ▁seeing ▁for ▁their ▁scenes - ▁and ▁Rick man ' s ▁scene ▁with ▁Hal"

In [25]:
toks200 = txts[:200].map(tkn)
toks200[0]

(#139) ['xxbos','xxmaj','alan','xxmaj','rickman','&','xxmaj','emma','xxmaj','thompson','give','good','performances','with','southern','/','xxmaj','new','xxmaj','orleans'...]

In [26]:
num = Numericalize()
num.setup(toks200)
coll_repr(num.vocab,20)

"(#1984) ['xxunk','xxpad','xxbos','xxeos','xxfld','xxrep','xxwrep','xxup','xxmaj','the','.',',','and','a','to','of','i','it','is','in'...]"

In [27]:
nums = num(toks)[:20]; nums

TensorText([  0,   0, 234,   0,   0, 199,  64, 731,  29,   0, 122,   0,   0, 943,  19,  20, 944, 294,  10,   0])

In [28]:
' '.join(num.vocab[o] for o in nums)

'xxunk xxunk & xxunk xxunk give good performances with xxunk / xxunk xxunk accents in this detective flick . xxunk'

In [29]:
stream = "In this chapter, we will go back over the example of classifying movie reviews we studied in chapter 1 and dig deeper under the surface. First we will look at the processing steps necessary to convert text into numbers and how to customize it. By doing this, we'll have another example of the PreProcessor used in the data block API.\nThen we will study how we build a language model and train it for a while."
tokens = tkn(stream)

In [30]:
tokens

(#90) ['xxbos','xxmaj','in','this','chapter',',','we','will','go','back','over','the','example','of','classifying','movie','reviews','we','studied','in'...]

In [31]:
bs,seq_len = 6,15
d_tokens = np.array([tokens[i*seq_len:(i+1)*seq_len] for i in range(bs)])
df = pd.DataFrame(d_tokens)
display(HTML(df.to_html(index=False,header=None)))

xxbos,xxmaj,in,this,chapter,",",we,will,go,back,over,the,example,of,classifying
movie,reviews,we,studied,in,chapter,1,and,dig,deeper,under,the,surface,.,xxmaj
first,we,will,look,at,the,processing,steps,necessary,to,convert,text,into,numbers,and
how,to,customize,it,.,xxmaj,by,doing,this,",",we,'ll,have,another,example
of,the,preprocessor,used,in,the,data,block,xxup,api,.,\n,xxmaj,then,we
will,study,how,we,build,a,language,model,and,train,it,for,a,while,.


In [32]:
nums200 = toks200.map(num)

In [34]:
#hide
nums200

(#200) [TensorText([   2,    8,    0,    8, 1442,  234,    8,    0,    8,    0,  199,   64,  731,   29,    0,  122,    8,  253,    8,    0,  943,   19,   20,  944,  294,   10,    8,   17,   25,  338,  408,
              28,  102,    0,   12,    8, 1442,   25,  160,   29,    8,    0,    8,    0,   10,    8,  163,  320,  164,    0,   14, 1443,  295,   77,  254,   61,    9,   27,   11,   17,  296,   10,
               8,    9,  110,   28,    9,   27,  650, 1134,   11,   31,   42,  321,    9,  732,   19,  152,    9,   32,   22,   17,   23,   36, 1135,  123,   33,  101,   33,   17,   81,   38,   87,
              10,    8,    9,  180,   22,   17,   18,    0,   49,   13,  264, 1444, 1445,   12,  835, 1444,    0,   76,    0,  188,   10,    8,    9,   27,   18,  338,   13,    0,   54,   28,  132,
              79,   90,  456,  731,   49,    8, 1442,   11,    8,    0,   11,   12,    8,    0,   10]),TensorText([   2,   16,   38,  140,   20,   27,   12,   16,   73,   36,  255,   28,   20,   27, 1

In [35]:
dl = LMDataLoader(nums200)

In [36]:
dl

In [37]:
x,y = first(dl)
x.shape,y.shape

(torch.Size([64, 72]), torch.Size([64, 72]))

In [38]:
' '.join(num.vocab[o] for o in x[0][:20])

'xxbos xxmaj xxunk xxmaj rickman & xxmaj xxunk xxmaj xxunk give good performances with xxunk / xxmaj new xxmaj xxunk'

In [39]:
' '.join(num.vocab[o] for o in y[0][:20])

'xxmaj xxunk xxmaj rickman & xxmaj xxunk xxmaj xxunk give good performances with xxunk / xxmaj new xxmaj xxunk accents'

In [40]:
get_imdb = partial(get_text_files, folders=['train', 'test', 'unsup'])

dls_lm = DataBlock(
    blocks=TextBlock.from_folder(path, is_lm=True),
    get_items=get_imdb, splitter=RandomSplitter(0.1)
).dataloaders(path, path=path, bs=128, seq_len=80)

In [41]:
dls_lm.show_batch(max_n=2)

xxbos xxmaj the book , while not particularly great , was decent , but this movie completely changes it . a lot of the elements of the story are consistent between the book and the movie , but xxmaj dr . xxmaj ross ' character goes from a creatively written character who lives for money and ends up causing the volcanic eruption with her greed to a heart - on - her - sleeve damsel in distress who wo n't
extent but like other comments have stated this is not for everyone . xxmaj there are cruel and upsetting scenes involving the young female characters and it would take a hardened heart to not to flinch or be moved by what the characters have to endure . xxmaj xxunk scientific experiments , limbs are torn off , there are scenes of implied sexual abuse . xxmaj but this is not some misogynistic experience of sadism , at its centre it
xxmaj the book , while not particularly great , was decent , but this movie completely changes it . a lot of the elements of the story are consistent between 

In [42]:
learn = language_model_learner(
    dls_lm, AWD_LSTM, drop_mult=0.3, 
    metrics=[accuracy, Perplexity()], cbs=[MixedPrecision()]).to_bf16()

In [60]:
learn.save('1epoch')

Path('/Users/aidardarmesh/.fastai/data/imdb/models/1epoch.pth')

In [ ]:
learn.unfreeze()
learn.fit_one_cycle(10, 2e-3)

/Users/aidardarmesh/epam/dl-practitioner/.venv/lib/python3.11/site-packages/torch/amp/autocast_mode.py:270: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
/Users/aidardarmesh/epam/dl-practitioner/.venv/lib/python3.11/site-packages/fastai/callback/fp16.py:47: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  self.autocast,self.learn.scaler,self.scales = autocast('cuda', dtype=dtype),GradScaler('cuda', **self.kwargs),L()


epoch,train_loss,valid_loss,accuracy,perplexity,time
